# Holt-Winters' Exponential Smoothing (Triple Exponential Smoothing)

- **SES** smooths the **level** only.
- **Holt** adds a **trend**.
- **Holt-Winters** adds a **seasonal** component too — so it handles series with **level + trend + seasonality**.

It uses **three** smoothing parameters: $\alpha$ (level), $\beta$ (trend), $\gamma$ (seasonality).

## 1. Additive vs Multiplicative

There are two flavours, depending on **how the seasonal swings behave** as the series grows:

| | **Additive** | **Multiplicative** |
|---|---|---|
| Trend? | Yes | Yes |
| Seasonality? | Yes, **constant** size | Yes, **proportional** size |
| Seasonal swings | stay the **same magnitude** even as the level rises | **grow** as the series trends upward |
| Example | Call volumes — hourly peak is a fixed *number* of extra calls | Quarterly retail sales — Q4 spike is a fixed *percentage* of a growing total |

**Rule of thumb:** if the seasonal "waves" get taller as the level increases → **multiplicative**; if they stay the same height → **additive**.

## 2. The components and equations

Three smoothed components, each updated every period:

$$\hat{L}_t\ \text{(level)}, \qquad \hat{T}_t\ \text{(trend)}, \qquad \hat{S}_t\ \text{(seasonality)}$$

**Forecast equations**

$$\hat{y}_{t+1} = \hat{L}_t + \hat{T}_t + \hat{S}_{t+1-m} \qquad \textbf{(additive)}$$
$$\hat{y}_{t+1} = (\hat{L}_t + \hat{T}_t)\,\hat{S}_{t+1-m} \qquad \textbf{(multiplicative)}$$

where each component is a function of the previous components, smoothed by its parameter:

$$\hat{L}_t = f(\dots)\ \text{smoothed by }\alpha, \quad \hat{T}_t = f(\dots)\ \text{smoothed by }\beta, \quad \hat{S}_t = f(\dots)\ \text{smoothed by }\gamma$$

> **Note:** $m$ is the **number of time steps in one season** (the seasonal period). In our example $m = 3$.

### The additive update equations in full

$$\hat{L}_t = \alpha\,(y_t - \hat{S}_{t-m}) + (1-\alpha)\,(\hat{L}_{t-1} + \hat{T}_{t-1}) \qquad \text{(level)}$$
$$\hat{T}_t = \beta\,(\hat{L}_t - \hat{L}_{t-1}) + (1-\beta)\,\hat{T}_{t-1} \qquad \text{(trend)}$$
$$\hat{S}_t = \gamma\,(y_t - \hat{L}_t) + (1-\gamma)\,\hat{S}_{t-m} \qquad \text{(seasonal)}$$

## 3. By-hand worked example (additive)

Same data as before, with $m = 3$ and $\alpha = \beta = \gamma = 0.2$:

| Month | Feb(1) | Mar(2) | Apr(3) | May(4) | Jun(5) | Jul(6) | Aug(7) | Sep(8) | Oct(9) | Nov(10) |
|---|---|---|---|---|---|---|---|---|---|---|
| **Actual** | 26 | 8 | 17 | 29 | 34 | 17 | 22 | 19 | 16 | 22 |

Train = Feb–Aug (1–7), test = Sep–Nov (8–10).

### Step 0 — initialise using the first full season (months 1–3)
We need a starting level, trend, and **one whole season** of seasonal indices before we can forecast.

$$\hat{L}_3 = \frac{y_1+y_2+y_3}{3} = \frac{26+8+17}{3} = 17, \qquad \hat{T}_3 = 0$$
$$\hat{S}_1 = y_1 - \hat{L}_3 = 26-17 = 9, \quad \hat{S}_2 = 8-17 = -9, \quad \hat{S}_3 = 17-17 = 0$$

### Step 1 — first forecast, month 4 (May)
$$\hat{y}_4 = \hat{L}_3 + \hat{T}_3 + \hat{S}_{4-3} = 17 + 0 + \hat{S}_1 = 17 + 9 = \mathbf{26}$$

### Step 2 — update at month 4 ($y_4 = 29$)
$$\hat{L}_4 = 0.2(y_4 - \hat{S}_1) + 0.8(\hat{L}_3+\hat{T}_3) = 0.2(29-9) + 0.8(17) = \mathbf{17.6}$$
$$\hat{T}_4 = 0.2(\hat{L}_4-\hat{L}_3) + 0.8(0) = 0.2(0.6) = \mathbf{0.12}$$
$$\hat{S}_4 = 0.2(y_4-\hat{L}_4) + 0.8\,\hat{S}_1 = 0.2(11.4) + 0.8(9) = \mathbf{9.48}$$
$$\hat{y}_5 = \hat{L}_4 + \hat{T}_4 + \hat{S}_{5-3} = 17.6 + 0.12 + \hat{S}_2 = 17.6 + 0.12 - 9 = \mathbf{8.72}$$

…and we keep rolling forward. The code below does all the bookkeeping.

## 4. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)

months = ["Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov"]
actual = [26, 8, 17, 29, 34, 17, 22, 19, 16, 22]
df = pd.DataFrame({"month": months, "actual": actual}, index=range(1, 11))
df.index.name = "t"
df.T

## 5. Run the additive recursion in code

In [ ]:
alpha = beta = gamma = 0.2
m = 3                       # season length
train_end = 7              # Aug
y = df["actual"].to_dict()

# --- initialise from the first full season (months 1..m) ---
L = {m: sum(y[i] for i in range(1, m + 1)) / m}    # L_3 = mean of first season = 17
T = {m: 0.0}
S = {i: y[i] - L[m] for i in range(1, m + 1)}       # S_1=9, S_2=-9, S_3=0
yhat = {}

# --- roll forward over the training set ---
for t in range(m + 1, train_end + 1):              # t = 4 .. 7
    yhat[t] = L[t-1] + T[t-1] + S[t-m]                              # forecast
    L[t] = alpha * (y[t] - S[t-m]) + (1 - alpha) * (L[t-1] + T[t-1])  # level
    T[t] = beta  * (L[t] - L[t-1]) + (1 - beta) * T[t-1]             # trend
    S[t] = gamma * (y[t] - L[t])  + (1 - gamma) * S[t-m]             # season

train_tbl = pd.DataFrame({
    "actual":   [y[t] for t in range(m, train_end + 1)],
    "L":        [L[t] for t in range(m, train_end + 1)],
    "T":        [T[t] for t in range(m, train_end + 1)],
    "S":        [S.get(t, np.nan) for t in range(m, train_end + 1)],
    "forecast": [np.nan] + [yhat[t] for t in range(m + 1, train_end + 1)],
}, index=range(m, train_end + 1)).round(2)
train_tbl

The `forecast` column gives **26, 8.72, 23.91, 32.86** for May–Aug — reproducing the slide's in-sample predictions (26, 8.72, 23.91; the slide shows 32.98 for Aug, a small hand-rounding difference).

## 6. Forecast the test months (multi-step)

From the last trained point (Aug, $t=7$) we forecast $h = 1,2,3$ steps ahead, re-using the most recent seasonal index for each position in the cycle:

$$\hat{y}_{7+h} = \hat{L}_7 + h\,\hat{T}_7 + \hat{S}_{7+h-m}$$

In [ ]:
test_preds = {7 + h: L[train_end] + h * T[train_end] + S[7 + h - m] for h in (1, 2, 3)}

test_df = pd.DataFrame({
    "month":    [months[t-1] for t in test_preds],
    "actual":   [y[t] for t in test_preds],
    "forecast": [round(v, 2) for v in test_preds.values()],
}, index=list(test_preds))
test_df

> **Heads-up — initialisation matters.** This from-scratch run gives test forecasts close to, but not identical to, the slide's (17.66, 20.63, 29.84). Holt-Winters is **very sensitive to how the initial level / trend / seasonal indices are set**, and different textbooks / libraries use different conventions. The *method* is the same; the exact seed differs. For the evaluation walk-through below we use the slide's stated predictions so the arithmetic lines up with the slide.

## 7. Evaluate on the test set — RMSE & MAPE

Using the slide's test predictions (Sep, Oct, Nov):

In [ ]:
act        = np.array([19, 16, 22], dtype=float)        # actual Sep, Oct, Nov
slide_pred = np.array([17.66, 20.63, 29.84])            # slide's predictions
err = act - slide_pred
n   = len(act)

sse  = np.sum(err ** 2)
rmse = np.sqrt(sse / n)
mape = np.mean(np.abs(err / act)) * 100

print(f"sum of squared errors      = {sse:.2f}")
print(f"RMSE = sqrt(SSE / n)       = {rmse:.2f}")
print(f"MAPE                       = {mape:.2f}%")

> **⚠️ Note on the slide's RMSE (same issue as the Holt slide).** The slide writes
> $$\text{RMSE} = \sqrt{\tfrac{(19-17.66)^2+(16-20.63)^2+(22-29.84)^2}{3}} = \sqrt{84.70} = 9.20$$
> Here **84.70 is the *sum* of the squared errors, not the sum ÷ 3** — so $\sqrt{84.70}=9.20$ skips the `/3`. The correct RMSE is
> $$\sqrt{84.70/3} = \sqrt{28.23} \approx \mathbf{5.31}.$$
> The **MAPE = 23.88%** on the slide is computed correctly.

## 8. Multiplicative variant & statsmodels

For real work, use `statsmodels` `ExponentialSmoothing`, which handles both variants and chooses good initial values and parameters automatically.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

train = np.array(actual[:7], dtype=float)

add_fit = ExponentialSmoothing(train, trend="add", seasonal="add",
                               seasonal_periods=3,
                               initialization_method="estimated").fit()
print("Additive 3-step forecast      :", np.round(add_fit.forecast(3), 2))

# Multiplicative needs strictly positive data (it does here)
mul_fit = ExponentialSmoothing(train, trend="add", seasonal="mul",
                               seasonal_periods=3,
                               initialization_method="estimated").fit()
print("Multiplicative 3-step forecast:", np.round(mul_fit.forecast(3), 2))

## 9. Summary

- **Holt-Winters = level + trend + seasonality**, smoothed by $\alpha, \beta, \gamma$ respectively.
- Choose **additive** when seasonal swings are a **constant size**, **multiplicative** when they **grow with the level**.
- $m$ = number of periods in one season; you need **at least one full season** of data to initialise the seasonal indices.
- Forecast adds (or multiplies) the seasonal index of the matching position in the cycle to the projected level + trend.
- Initialisation strongly affects the exact numbers — rely on a library for production, use the by-hand recursion to understand *why* it works.
- Evaluate with **RMSE** (remember the `÷ n` — don't repeat the slide's slip) and **MAPE**.